# Free GPU agent worker

Run a JSONL job file through a small open model on whatever machine executes
this notebook.

> **Remote GPU warning.** If this notebook is running on Kaggle or Colab,
> `torch.cuda` reports the remote T4/P100 (or another rented accelerator), **not
> this local Windows machine**. The local host is assumed to have 16 GB RAM and
> no NVIDIA CUDA GPU. A remote free GPU is a different computer with different
> memory, different libraries and a different lifetime.

No API key, token or secret is used by this notebook.


## 1. Inspect the machine

This cell is read-only. It prints the CPU, RAM and any visible CUDA GPU. It
works before `transformers` is installed.


In [ ]:
import json
import os
import platform
import shutil
import subprocess
import sys

print("python:", sys.version.split()[0])
print("platform:", platform.platform())
print("machine:", platform.machine())
print("cpu_count:", os.cpu_count())

try:
    import psutil
    memory = psutil.virtual_memory()
    print(f"ram: {memory.total / 1024**3:.1f} GiB total, {memory.available / 1024**3:.1f} GiB available")
except Exception:
    print("ram: psutil not installed (pip install psutil for a reading)")

print("\n--- nvidia-smi ---")
nvidia_smi = shutil.which("nvidia-smi")
if nvidia_smi:
    result = subprocess.run(
        [nvidia_smi, "--query-gpu=name,memory.total,memory.used,driver_version", "--format=csv,noheader"],
        capture_output=True,
        text=True,
    )
    if result.returncode == 0:
        print(result.stdout.strip())
    else:
        print("nvidia-smi failed:", result.stderr.strip())
else:
    print("nvidia-smi not found")

print("\n--- torch.cuda ---")
try:
    import torch
    print("torch:", torch.__version__)
    print("cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        for index in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(index)
            print(
                f"gpu[{index}]: {props.name}, "
                f"{props.total_memory / 1024**3:.1f} GiB VRAM, "
                f"compute capability {props.major}.{props.minor}"
            )
    else:
        print("No CUDA device is visible to torch. CPU inference is slow but a 0.5B model can still run.")
except Exception as exc:
    print(f"torch not importable ({type(exc).__name__}): {exc}")


## 2. Configure the job

Defaults are intentionally small: a 0.5B instruct model, greedy decoding and a
256-token output cap. Override any value with an environment variable before
running the cell.

`system` is optional per job; when absent the notebook-level system prompt is
used.


In [ ]:
from pathlib import Path

JOB_FILE = Path(os.environ.get("AGENT_JOB_FILE", "jobs.jsonl")).expanduser()
OUT_FILE = Path(os.environ.get("AGENT_OUT_FILE", "results.jsonl")).expanduser()

# Small open model. Qwen2.5-0.5B-Instruct fits a free T4 and can run on CPU slowly.
MODEL_ID = os.environ.get("AGENT_MODEL_ID", "Qwen/Qwen2.5-0.5B-Instruct")
BACKEND = os.environ.get("AGENT_BACKEND", "transformers")  # "transformers" or "ollama"

OLLAMA_MODEL = os.environ.get("AGENT_OLLAMA_MODEL", "qwen2.5:0.5b")
OLLAMA_URL = os.environ.get("AGENT_OLLAMA_URL", "http://127.0.0.1:11434")

MAX_NEW_TOKENS = int(os.environ.get("AGENT_MAX_NEW_TOKENS", "256"))
SYSTEM_PROMPT = os.environ.get(
    "AGENT_SYSTEM_PROMPT",
    "You are a careful coding assistant. Answer concisely and do not invent facts.",
)
SEED = 42

print("job file:", JOB_FILE.resolve())
print("output file:", OUT_FILE.resolve())
print("backend:", BACKEND)
print("model:", MODEL_ID if BACKEND == "transformers" else OLLAMA_MODEL)
print("max_new_tokens:", MAX_NEW_TOKENS)
print("seed:", SEED)


## 3. Create or inspect the JSONL job file

Expected format, one object per line:

```json
{"id": "job-1", "prompt": "What is 17 + 25?", "system": "Reply tersely."}
{"id": "job-2", "prompt": "Summarise this function."}
```

`id` must be unique and non-empty. `prompt` must be non-empty. `system` is
optional and must be a string when present.


In [ ]:
SAMPLE_JOBS = [
    {
        "id": "demo-1",
        "system": "You are terse.",
        "prompt": "What is 17 + 25? Reply with just the number.",
    },
    {
        "id": "demo-2",
        "prompt": "Name one reason to write results incrementally.",
    },
]

if not JOB_FILE.exists():
    JOB_FILE.write_text(
        "".join(json.dumps(job, ensure_ascii=False) + "\n" for job in SAMPLE_JOBS),
        encoding="utf-8",
    )
    print(f"wrote sample {JOB_FILE}")
else:
    print(f"using existing {JOB_FILE}")

print(JOB_FILE.read_text(encoding="utf-8")[:1200])


## 4. Optional dependency install

The cell is a no-op unless you set `INSTALL_DEPENDENCIES = True` and run it.
This keeps a plain notebook open from installing anything by accident.

For the Ollama backend instead, install Ollama on the remote machine, start
`ollama serve`, and run `ollama pull qwen2.5:0.5b`. Ollama is not required for
the default transformers backend.


In [ ]:
# Change to True only when you want this cell to install packages.
INSTALL_DEPENDENCIES = False

if INSTALL_DEPENDENCIES:
    import sys
    import subprocess

    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "transformers>=4.46",
            "accelerate>=1.0",
            "torch>=2.4",
        ]
    )
    print("installed transformers, accelerate and torch")
else:
    print("install skipped; set INSTALL_DEPENDENCIES = True and run this cell only if imports fail")


## 5. Load the model

Runs only when you execute the cell. For the transformers backend the model is
loaded in fp16 on CUDA, or fp32 on CPU. For Ollama, nothing is loaded into this
Python process; the local HTTP service does the work.


In [ ]:
tokenizer = None
model = None

if BACKEND == "transformers":
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None,
        low_cpu_mem_usage=True,
    )
    model.eval()
    print("loaded", MODEL_ID)
    print("device:", next(model.parameters()).device)
    print("dtype:", next(model.parameters()).dtype)
elif BACKEND == "ollama":
    print("Ollama backend selected; no model is loaded into this process.")
    print("Make sure `ollama serve` is running and the model is pulled:", OLLAMA_MODEL)
else:
    raise SystemExit(f"unknown BACKEND: {BACKEND!r}")


## 6. Deterministic generation

Settings are greedy: `do_sample=False`, one beam, no repetition penalty, fixed
seed for the Ollama path. The same input should produce the same output for a
given model and runtime, up to library and hardware differences.

The Ollama path speaks to `127.0.0.1` only and needs no credential.


In [ ]:
def generate_transformers(prompt, system=None):
    import torch

    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    if getattr(tokenizer, "chat_template", None):
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    else:
        prefix = f"System: {system}\n" if system else ""
        text = f"{prefix}User: {prompt}\nAssistant:"

    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=1,
            repetition_penalty=1.0,
            pad_token_id=tokenizer.pad_token_id,
        )
    generated = output[0][inputs["input_ids"].shape[-1] :]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


def generate_ollama(prompt, system=None):
    import urllib.request

    payload = {
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "system": system or SYSTEM_PROMPT,
        "stream": False,
        "options": {
            "temperature": 0,
            "seed": SEED,
            "num_predict": MAX_NEW_TOKENS,
        },
    }
    request = urllib.request.Request(
        f"{OLLAMA_URL.rstrip('/')}/api/generate",
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(request, timeout=600) as response:
        body = json.loads(response.read().decode("utf-8"))
    return body.get("response", "").strip()


def generate(prompt, system=None):
    if BACKEND == "transformers":
        return generate_transformers(prompt, system)
    if BACKEND == "ollama":
        return generate_ollama(prompt, system)
    raise RuntimeError(f"unknown backend {BACKEND!r}")


## 7. Run jobs and write results incrementally

Every result is appended, flushed and `fsync`-ed before the next job starts. If
the session dies at job 40 of 100, the first 40 lines are already on disk.
Re-running skips ids already present in the output file.

A failed job is recorded with `"ok": false` and an error string, and the runner
continues to the next job.


In [ ]:
import os
import time


def read_jsonl(path):
    jobs = []
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, 1):
            line = line.strip()
            if not line:
                continue
            try:
                item = json.loads(line)
            except json.JSONDecodeError as exc:
                print(f"skip line {line_number}: invalid JSON ({exc})")
                continue
            if not isinstance(item, dict):
                print(f"skip line {line_number}: expected a JSON object")
                continue
            job_id = item.get("id")
            prompt = item.get("prompt")
            system = item.get("system")
            if not isinstance(job_id, str) or not job_id.strip():
                print(f"skip line {line_number}: id must be a non-empty string")
                continue
            if not isinstance(prompt, str) or not prompt.strip():
                print(f"skip line {line_number}: prompt must be a non-empty string")
                continue
            if system is not None and not isinstance(system, str):
                print(f"skip line {line_number}: system must be a string when present")
                continue
            jobs.append({"id": job_id, "prompt": prompt, "system": system})
    return jobs


def completed_ids(path):
    ids = set()
    if not path.exists():
        return ids
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            try:
                item = json.loads(line)
            except json.JSONDecodeError:
                continue
            if isinstance(item, dict) and isinstance(item.get("id"), str):
                ids.add(item["id"])
    return ids


jobs = read_jsonl(JOB_FILE)
done = completed_ids(OUT_FILE)
pending = [job for job in jobs if job["id"] not in done]
print(f"{len(jobs)} job(s), {len(done)} already complete, {len(pending)} pending")

OUT_FILE.parent.mkdir(parents=True, exist_ok=True)
with OUT_FILE.open("a", encoding="utf-8") as out:
    for position, job in enumerate(pending, 1):
        started = time.time()
        print(f"[{position}/{len(pending)}] {job['id']} ...", flush=True)
        try:
            text = generate(job["prompt"], job.get("system") or SYSTEM_PROMPT)
            record = {
                "id": job["id"],
                "model": MODEL_ID if BACKEND == "transformers" else OLLAMA_MODEL,
                "backend": BACKEND,
                "text": text,
                "seconds": round(time.time() - started, 3),
                "ok": True,
            }
        except Exception as exc:
            record = {
                "id": job["id"],
                "model": MODEL_ID if BACKEND == "transformers" else OLLAMA_MODEL,
                "backend": BACKEND,
                "error": f"{type(exc).__name__}: {exc}",
                "seconds": round(time.time() - started, 3),
                "ok": False,
            }
        out.write(json.dumps(record, ensure_ascii=False) + "\n")
        out.flush()
        os.fsync(out.fileno())
        print("  ok" if record["ok"] else f"  failed: {record['error']}", flush=True)

print("results:", OUT_FILE.resolve())


## 8. Exact runtime report

Keep this output with the results. It says which model, dtype, device and
library versions produced the file. It also repeats the remote-GPU warning,
because a Kaggle/Colab T4 is not the local workstation.


In [ ]:
print("=== exact runtime ===")
print("backend:", BACKEND)
if BACKEND == "transformers":
    import torch
    import transformers

    print("model_id:", MODEL_ID)
    print("model_commit:", getattr(model.config, "_commit_hash", None))
    print("model_dtype:", next(model.parameters()).dtype)
    print("model_device:", next(model.parameters()).device)
    print("transformers:", transformers.__version__)
    print("torch:", torch.__version__)
    print("cuda_available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        print("gpu:", props.name)
        print("vram_gib:", round(props.total_memory / 1024**3, 1))
else:
    print("ollama_model:", OLLAMA_MODEL)
    print("ollama_url:", OLLAMA_URL)
    print("torch/transformers were not imported for the Ollama backend.")

print("decoding: do_sample=False, num_beams=1, seed=42")
print()
print("REMOTE GPU REMINDER: a Kaggle/Colab GPU is not the local Windows machine.")
print("Local host assumption: 16 GB RAM, no NVIDIA CUDA GPU.")


## 9. Bring the results back

The output is plain JSONL at `results.jsonl` (or `AGENT_OUT_FILE`).

- **Kaggle:** `kaggle kernels output <owner>/<kernel> -p ./out` after the kernel
  run finishes. Or download from the notebook Output tab.
- **Colab:** download `results.jsonl` from the Files pane, or copy it to Google
  Drive from a cell you run yourself.
- **Lightning / Studio Lab:** download from the file browser or copy to your
  own storage.

Security: results may contain model output derived from your prompts. Treat
them like the prompts themselves. Do not put credentials in `jobs.jsonl`, and do
not echo environment variables into results.
